# ApprenticeOps — wave analysis (live snapshot)

Re-runnable analysis over a **committed snapshot** of the benchmark run.

Two views:
1. **Burn-through time per bracket** — stacked columns of wall-clock minutes (stacked by model). Bigger models cost more; this shows roughly how much longer a heavier bracket takes to grind through.
2. **Precision @ 0.7** — stacked columns of pass/fail per bracket, using a deterministic-score threshold of **0.7**.

The snapshot lives at `data/snapshots/results_snapshot.csv` (text-free: model, bracket, scenario, rep, det_score, decode_tok_s, wall_s, dnf, finish_reason). Refresh it from the run node any time (see the last cell).

In [ ]:
# Dependencies (no-op if already installed)
%pip install -q pandas matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PRECISION_THRESHOLD = 0.7  # deterministic-score pass bar
BRACKET_ORDER = ['0-1B', '1-2B', '2-3B', '3-4B', '4-5GB']

# Find the committed snapshot regardless of where the kernel started.
candidates = [
    Path('data/snapshots/results_snapshot.csv'),
    Path('../data/snapshots/results_snapshot.csv'),
    Path('../../data/snapshots/results_snapshot.csv'),
]
snapshot = next((p for p in candidates if p.exists()), None)
if snapshot is None:
    raise FileNotFoundError('results_snapshot.csv not found; run the refresh cell at the bottom.')
print('Loading', snapshot.resolve())

df = pd.read_csv(snapshot)
for c in ['det_score', 'decode_tok_s', 'wall_s', 'rep']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['bracket'] = pd.Categorical(df['bracket'], categories=BRACKET_ORDER, ordered=True)
present = [b for b in BRACKET_ORDER if (df['bracket'] == b).any()]
print(f'rows={len(df)}  models={df["model"].nunique()}  brackets={present}')
df.head()

## 1. Burn-through time per bracket (stacked by model)

Column height = total wall-clock minutes spent in that bracket so far; each segment is one model. The annotation shows the **per-model average**, which is the apples-to-apples cost (brackets can have different model counts).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
avg_per_model = {}
for i, b in enumerate(present):
    sub = df[df['bracket'] == b].groupby('model')['wall_s'].sum().sort_values(ascending=False)
    bottom = 0.0
    for model, val in sub.items():
        ax.bar(i, val / 60, bottom=bottom / 60, width=0.62, edgecolor='white', linewidth=0.5)
        bottom += val
    n_models = max(len(sub), 1)
    avg_per_model[b] = (bottom / 60) / n_models
    ax.text(i, bottom / 60 + max(bottom / 60 * 0.01, 1),
            f'{bottom/60:.0f} min\n{n_models} mdl · {avg_per_model[b]:.0f}/mdl',
            ha='center', va='bottom', fontsize=9)
ax.set_xticks(range(len(present)))
ax.set_xticklabels(present)
ax.set_ylabel('wall-clock minutes (stacked by model)')
ax.set_title('Burn-through time per bracket (stacked by model)')
ax.margins(y=0.12)
plt.tight_layout()
plt.show()

# Quick ratio sense-check (e.g. 1-2B vs 3-4B)
if '1-2B' in avg_per_model and '3-4B' in avg_per_model and avg_per_model['1-2B']:
    r = avg_per_model['3-4B'] / avg_per_model['1-2B']
    print(f"avg/model: 1-2B={avg_per_model['1-2B']:.1f} min, 3-4B={avg_per_model['3-4B']:.1f} min  ->  3-4B is {r:.1f}x slower")

## 2. Precision @ 0.7 per bracket (stacked pass / fail)

Each (model × scenario × rep) result counts as **pass** if its deterministic score ≥ 0.7, else **fail**. Columns are stacked pass+fail; the annotation is the pass-rate.

In [ ]:
passes, fails, rates = [], [], []
for b in present:
    sub = df[(df['bracket'] == b) & df['det_score'].notna()]
    p = int((sub['det_score'] >= PRECISION_THRESHOLD).sum())
    t = int(len(sub))
    passes.append(p)
    fails.append(t - p)
    rates.append(p / t if t else 0.0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(present, passes, width=0.62, label=f'pass (det ≥ {PRECISION_THRESHOLD})', color='#2e7d32')
ax.bar(present, fails, bottom=passes, width=0.62, label=f'fail (< {PRECISION_THRESHOLD})', color='#c62828')
for i, b in enumerate(present):
    total = passes[i] + fails[i]
    ax.text(i, total + max(total * 0.01, 0.5), f'{rates[i]*100:.0f}% pass', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('result count (model × scenario × rep)')
ax.set_title(f'Deterministic precision @ {PRECISION_THRESHOLD} per bracket (stacked pass/fail)')
ax.legend()
ax.margins(y=0.12)
plt.tight_layout()
plt.show()

## Refresh the snapshot

Re-pull the current run from the node into the committed CSV, then re-run the cells above. Run this from the repo root in a terminal:

```bash
ssh dragos@home-ai.hont.ro 'python3 - <<"PY"
import json,sys,csv
out=csv.writer(sys.stdout)
out.writerow(["model","bracket","scenario","rep","det_score","decode_tok_s","wall_s","dnf","finish_reason"])
for line in open("/tmp/sme-var/results.var.jsonl"):
    line=line.strip()
    if not line: continue
    try: d=json.loads(line)
    except: continue
    fr=d.get("gen_ai.response.finish_reasons")
    if isinstance(fr,list): fr=",".join(map(str,fr))
    out.writerow([d.get("model"),d.get("bracket"),d.get("scenario"),d.get("rep"),d.get("det_score"),d.get("decode_tok_s"),d.get("wall_s"),d.get("dnf"),fr])
PY' > data/snapshots/results_snapshot.csv
```